# 模型性能評估與視覺化

這個 notebook 詳細展示模型的各項性能指標和視覺化分析。

## 內容
- 載入訓練好的模型
- 混淆矩陣分析
- ROC 曲線和 AUC
- 錯誤分析
- 特徵重要性
- 測試案例展示

## 1. 匯入套件並載入模型

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import sys
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, roc_auc_score,
    precision_recall_curve, average_precision_score
)

# 設定繪圖樣式
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang TC']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 8)
sns.set_palette("husl")

# 加入 src 路徑
sys.path.append('../src')

print("✅ 套件匯入完成")

In [ ]:
# 載入模型和向量化器
with open('../models/model.pkl', 'rb') as f:
    model = pickle.load(f)
print("✅ 模型已載入")

with open('../models/vectorizer.pkl', 'rb') as f:
    vectorizer = pickle.load(f)
print("✅ 向量化器已載入")

print(f"\n模型類型: {type(model).__name__}")
print(f"特徵數量: {len(vectorizer.get_feature_names_out())}")

## 2. 準備測試資料

In [ ]:
from preprocessing import TextPreprocessor

# 載入資料
df = pd.read_csv('../data/sms_spam_no_header.csv', encoding='latin-1')
df['label'] = df['label'].astype(str)
df['message'] = df['message'].astype(str)

# 預處理
preprocessor = TextPreprocessor()
df['processed_message'] = df['message'].apply(preprocessor.preprocess)
df = df[df['processed_message'].str.strip() != '']

# 分割資料
X = df['processed_message']
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 向量化
X_test_tfidf = vectorizer.transform(X_test)

# 預測
y_pred = model.predict(X_test_tfidf)
y_pred_proba = model.predict_proba(X_test_tfidf)

print(f"測試集大小: {len(X_test)}")
print("✅ 資料準備完成")

## 3. 混淆矩陣

In [ ]:
# 計算混淆矩陣
cm = confusion_matrix(y_test, y_pred, labels=['ham', 'spam'])

# 視覺化混淆矩陣
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['HAM', 'SPAM'],
            yticklabels=['HAM', 'SPAM'],
            cbar_kws={'label': '數量'})
plt.title('混淆矩陣 (Confusion Matrix)', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('實際標籤', fontsize=12)
plt.xlabel('預測標籤', fontsize=12)

# 加入說明文字
tn, fp, fn, tp = cm.ravel()
plt.text(0.5, -0.15, 
         f'True Negative: {tn} | False Positive: {fp}\nFalse Negative: {fn} | True Positive: {tp}',
         ha='center', transform=plt.gca().transAxes, fontsize=10)

plt.tight_layout()
plt.show()

print(f"\n混淆矩陣分析:")
print(f"True Negative (正確預測為 HAM):  {tn}")
print(f"False Positive (誤判為 SPAM):    {fp}")
print(f"False Negative (漏判 SPAM):      {fn}")
print(f"True Positive (正確預測為 SPAM): {tp}")

## 4. ROC 曲線和 AUC

In [ ]:
# 計算 ROC 曲線
# 取得 SPAM 類別的機率
spam_idx = list(model.classes_).index('spam')
y_test_binary = (y_test == 'spam').astype(int)
y_score = y_pred_proba[:, spam_idx]

fpr, tpr, thresholds = roc_curve(y_test_binary, y_score)
roc_auc = auc(fpr, tpr)

# 繪製 ROC 曲線
plt.figure(figsize=(10, 8))
plt.plot(fpr, tpr, color='#e74c3c', lw=3, 
         label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='#95a5a6', lw=2, linestyle='--', 
         label='Random Classifier (AUC = 0.5000)')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (偽陽性率)', fontsize=12)
plt.ylabel('True Positive Rate (真陽性率)', fontsize=12)
plt.title('ROC 曲線 (Receiver Operating Characteristic)', 
          fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n📊 ROC AUC Score: {roc_auc:.4f}")
print(f"\n解釋: AUC 值越接近 1，模型效果越好。")
print(f"目前模型 AUC = {roc_auc:.4f}，表示模型有優秀的分類能力。")

## 5. Precision-Recall 曲線

In [ ]:
# 計算 Precision-Recall 曲線
precision, recall, pr_thresholds = precision_recall_curve(y_test_binary, y_score)
avg_precision = average_precision_score(y_test_binary, y_score)

# 繪製曲線
plt.figure(figsize=(10, 8))
plt.plot(recall, precision, color='#3498db', lw=3, 
         label=f'PR curve (AP = {avg_precision:.4f})')
plt.xlabel('Recall (召回率)', fontsize=12)
plt.ylabel('Precision (精確率)', fontsize=12)
plt.title('Precision-Recall 曲線', fontsize=14, fontweight='bold')
plt.legend(loc="lower left", fontsize=11)
plt.grid(True, alpha=0.3)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.tight_layout()
plt.show()

print(f"\n📊 Average Precision Score: {avg_precision:.4f}")

## 6. 詳細分類報告

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 計算各項指標
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label='spam')
recall = recall_score(y_test, y_pred, pos_label='spam')
f1 = f1_score(y_test, y_pred, pos_label='spam')

# 創建表格
metrics_df = pd.DataFrame({
    '指標': ['Accuracy (準確率)', 'Precision (精確率)', 'Recall (召回率)', 'F1 Score', 'ROC AUC'],
    '分數': [accuracy, precision, recall, f1, roc_auc],
    '百分比': [f'{accuracy*100:.2f}%', f'{precision*100:.2f}%', 
              f'{recall*100:.2f}%', f'{f1*100:.2f}%', f'{roc_auc*100:.2f}%']
})

print("\n" + "="*60)
print("模型性能指標總覽")
print("="*60)
print(metrics_df.to_string(index=False))
print("="*60)

# 視覺化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 長條圖
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']
bars = ax1.bar(metrics_df['指標'], metrics_df['分數'], color=colors)
ax1.set_ylim(0, 1.1)
ax1.set_ylabel('分數', fontsize=12)
ax1.set_title('各項性能指標', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)
ax1.tick_params(axis='x', rotation=15)

for bar, value in zip(bars, metrics_df['分數']):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 0.02,
             f'{value:.4f}', ha='center', va='bottom', fontweight='bold')

# 雷達圖
categories = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']
values = [accuracy, precision, recall, f1, roc_auc]
values += values[:1]  # 閉合圖形

angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
angles += angles[:1]

ax2 = plt.subplot(122, projection='polar')
ax2.plot(angles, values, 'o-', linewidth=2, color='#e74c3c')
ax2.fill(angles, values, alpha=0.25, color='#e74c3c')
ax2.set_xticks(angles[:-1])
ax2.set_xticklabels(categories)
ax2.set_ylim(0, 1)
ax2.set_title('性能指標雷達圖', fontsize=14, fontweight='bold', pad=20)
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# 分類報告
print("\n詳細分類報告:")
print(classification_report(y_test, y_pred, target_names=['HAM', 'SPAM']))

## 7. 錯誤分析

In [ ]:
# 找出錯誤預測的案例
X_test_list = X_test.tolist()
y_test_list = y_test.tolist()
errors_df = pd.DataFrame({
    'message': X_test_list,
    'actual': y_test_list,
    'predicted': y_pred
})

errors_df = errors_df[errors_df['actual'] != errors_df['predicted']]

print(f"\n總共有 {len(errors_df)} 個錯誤預測 (錯誤率: {len(errors_df)/len(y_test)*100:.2f}%)")
print(f"\n錯誤類型分布:")
print(errors_df.groupby(['actual', 'predicted']).size())

In [ ]:
# 顯示 False Positives (誤判為 SPAM)
false_positives = errors_df[errors_df['predicted'] == 'spam']
print(f"\n❌ False Positives (誤判為 SPAM): {len(false_positives)} 個\n")
print("範例:")
for idx, row in false_positives.head(5).iterrows():
    print(f"- {row['message'][:100]}...")
    print()

In [ ]:
# 顯示 False Negatives (漏判 SPAM)
false_negatives = errors_df[errors_df['predicted'] == 'ham']
print(f"\n❌ False Negatives (漏判 SPAM): {len(false_negatives)} 個\n")
print("範例:")
for idx, row in false_negatives.head(5).iterrows():
    print(f"- {row['message'][:100]}...")
    print()

## 8. 特徵重要性分析

In [ ]:
# 取得特徵名稱和 log 機率
feature_names = vectorizer.get_feature_names_out()
spam_idx = list(model.classes_).index('spam')
ham_idx = list(model.classes_).index('ham')

# 計算特徵的 log 機率差異
log_prob_spam = model.feature_log_prob_[spam_idx]
log_prob_ham = model.feature_log_prob_[ham_idx]
log_prob_diff = log_prob_spam - log_prob_ham

# 取得最具代表性的特徵
top_spam_indices = log_prob_diff.argsort()[-20:][::-1]
top_ham_indices = log_prob_diff.argsort()[:20]

print("\n📧 最具 SPAM 特徵的詞 (Top 20):")
print("-" * 50)
for i, idx in enumerate(top_spam_indices, 1):
    print(f"{i:2d}. {feature_names[idx]:15s} (差異: {log_prob_diff[idx]:.4f})")

print("\n📨 最具 HAM 特徵的詞 (Top 20):")
print("-" * 50)
for i, idx in enumerate(top_ham_indices, 1):
    print(f"{i:2d}. {feature_names[idx]:15s} (差異: {log_prob_diff[idx]:.4f})")

In [ ]:
# 視覺化特徵重要性
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# SPAM 特徵
spam_words = [feature_names[i] for i in top_spam_indices[:10]]
spam_scores = [log_prob_diff[i] for i in top_spam_indices[:10]]
ax1.barh(spam_words, spam_scores, color='#e74c3c')
ax1.set_xlabel('Log 機率差異', fontsize=12)
ax1.set_title('Top 10 SPAM 特徵詞', fontsize=14, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)

# HAM 特徵
ham_words = [feature_names[i] for i in top_ham_indices[:10]]
ham_scores = [log_prob_diff[i] for i in top_ham_indices[:10]]
ax2.barh(ham_words, ham_scores, color='#2ecc71')
ax2.set_xlabel('Log 機率差異', fontsize=12)
ax2.set_title('Top 10 HAM 特徵詞', fontsize=14, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## 9. 測試案例展示

In [ ]:
# 測試案例
test_cases = [
    ("Congratulations! You've won $1000! Click here to claim your prize now!", "SPAM"),
    ("Hi, are you free for lunch tomorrow?", "HAM"),
    ("FREE FREE FREE! Limited time offer! Click now!", "SPAM"),
    ("WINNER! As a valued customer, you have been selected to receive a cash prize!", "SPAM"),
    ("Your order #12345 has been shipped and will arrive tomorrow.", "HAM"),
    ("Meeting rescheduled to 3pm in conference room B", "HAM"),
    ("URGENT! Your account will be closed unless you verify your information immediately!", "SPAM"),
    ("Thanks for the dinner last night. Let's do it again soon!", "HAM")
]

print("\n" + "="*80)
print("測試案例預測結果")
print("="*80)

results = []
for msg, expected in test_cases:
    # 預處理
    processed = preprocessor.preprocess(msg)
    # 向量化
    vectorized = vectorizer.transform([processed])
    # 預測
    prediction = model.predict(vectorized)[0]
    proba = model.predict_proba(vectorized)[0]
    spam_prob = proba[spam_idx]
    ham_prob = proba[ham_idx]
    
    result = "✅" if prediction.upper() == expected else "❌"
    results.append({
        '訊息': msg[:60] + '...' if len(msg) > 60 else msg,
        '預期': expected,
        '預測': prediction.upper(),
        'SPAM機率': f'{spam_prob*100:.2f}%',
        '結果': result
    })
    
    print(f"\n訊息: {msg}")
    print(f"預期: {expected} | 預測: {prediction.upper()} | SPAM機率: {spam_prob*100:.2f}% | {result}")
    print("-" * 80)

# 創建結果表格
results_df = pd.DataFrame(results)
print("\n\n測試結果總覽:")
print(results_df.to_string(index=False))

accuracy = sum(1 for r in results if r['結果'] == '✅') / len(results)
print(f"\n測試準確率: {accuracy*100:.2f}% ({sum(1 for r in results if r['結果'] == '✅')}/{len(results)})")

## 10. 總結報告

In [ ]:
# 生成總結報告
summary = f"""
{'='*80}
模型性能評估總結報告
{'='*80}

📊 基本資訊
  - 模型類型: {type(model).__name__}
  - 特徵提取: TF-IDF (max_features={len(vectorizer.get_feature_names_out())})
  - 測試集大小: {len(y_test)} 筆

📈 性能指標
  - 準確率 (Accuracy):   {accuracy*100:.2f}%
  - 精確率 (Precision):  {precision*100:.2f}%
  - 召回率 (Recall):     {recall*100:.2f}%
  - F1 分數:            {f1*100:.2f}%
  - ROC AUC:            {roc_auc:.4f}

🔍 混淆矩陣
  - True Negative (正確預測為 HAM):   {tn}
  - False Positive (誤判為 SPAM):     {fp}
  - False Negative (漏判 SPAM):       {fn}
  - True Positive (正確預測為 SPAM):  {tp}

❌ 錯誤分析
  - 總錯誤數: {len(errors_df)}
  - 錯誤率: {len(errors_df)/len(y_test)*100:.2f}%
  - False Positives: {len(false_positives)}
  - False Negatives: {len(false_negatives)}

✅ 結論
  模型在測試集上表現優異，準確率達 {accuracy*100:.2f}%，精確率達 {precision*100:.2f}%。
  ROC AUC 為 {roc_auc:.4f}，顯示模型具有優秀的分類能力。
  適合用於垃圾郵件分類任務。

{'='*80}
"""

print(summary)

# 儲存報告
with open('../models/evaluation_report.txt', 'w', encoding='utf-8') as f:
    f.write(summary)
print("\n✅ 評估報告已儲存: ../models/evaluation_report.txt")

## 完成！

這個 notebook 展示了完整的模型性能評估流程，包括：

✅ 混淆矩陣分析  
✅ ROC 曲線和 AUC  
✅ Precision-Recall 曲線  
✅ 詳細分類報告  
✅ 錯誤分析  
✅ 特徵重要性  
✅ 測試案例展示  
✅ 總結報告

模型表現優異，可以部署使用！